# 05 - Hidden-feature steering and method comparison

This note moves the intervention from 2D sample space into a hidden
layer of the unconditional velocity network.

We collect hidden features from labeled examples after forward noising, learn a
target-vs-rest direction offline, and add that direction during sampling. No
gradient is computed at inference.

**Learning goals**

1. Measure when hidden activations become class-informative.
2. Learn and visualize a class direction in feature space.
3. Test whether one direction transfers across nearby timesteps.
4. Compare baseline, noise alignment, gradient guidance, activation steering,
   and a two-stage combination under paired conditions.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


In [ ]:
model, losses, trained_now = load_or_train_model(device=device)
reference_data, reference_labels = sample_labeled_mixture(30000, device=device)
stats = estimate_gaussian_stats(reference_data, reference_labels)
TARGET_CLASS = 2
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
checkpoint_digest_before = model_state_digest(model)
print("frozen checkpoint digest:", checkpoint_digest_before[:16])


## 1. When is class information visible in the hidden layer?

For each time, we draw separate balanced training and test datasets, corrupt
them with separate noise draws, and record one hidden layer. A ridge probe is
fit only on the training activations and scored only on the untouched test
activations. Twenty shuffled-training-label probes provide an empirical null.
Their mean is usually near the one-in-three chance rate, but an individual
shuffle can score higher by accidentally assigning output names to separable
clusters. We therefore compare the true probe with the null's 95th percentile,
not with the single largest shuffle. No probe is used to train the generator.


In [ ]:
def wilson_interval(successes, total, z=1.96):
    proportion = successes / total
    denominator = 1 + z**2 / total
    center = (proportion + z**2 / (2 * total)) / denominator
    radius = z * np.sqrt(proportion * (1 - proportion) / total + z**2 / (4 * total**2)) / denominator
    return center - radius, center + radius

probe_rows = []
directions = {}
probes = {}
feature_sets = {}
for time_index, t_value in enumerate([0.10, 0.25, 0.40, 0.55, 0.70, 0.85]):
    set_seed(SEED + 500 + 2 * time_index)
    train_features, train_labels = collect_forward_activations(
        model, t_value=t_value, n_per_class=600
    )
    set_seed(SEED + 501 + 2 * time_index)
    test_features, test_labels = collect_forward_activations(
        model, t_value=t_value, n_per_class=400
    )

    probe = fit_linear_probe(train_features, train_labels)
    train_accuracy = (probe.predict(train_features) == train_labels).float().mean().item()
    test_predictions = probe.predict(test_features)
    test_successes = int((test_predictions == test_labels).sum().item())
    test_accuracy = test_successes / test_labels.numel()
    interval_low, interval_high = wilson_interval(test_successes, test_labels.numel())

    shuffled_accuracies = []
    for null_index in range(20):
        set_seed(SEED + 2000 + 100 * time_index + null_index)
        shuffled_labels = train_labels[torch.randperm(train_labels.numel(), device=device)]
        shuffled_probe = fit_linear_probe(train_features, shuffled_labels)
        shuffled_accuracy = (
            shuffled_probe.predict(test_features) == test_labels
        ).float().mean().item()
        shuffled_accuracies.append(shuffled_accuracy)

    direction = discriminant_direction(train_features, train_labels, TARGET_CLASS)
    directions[t_value] = direction
    probes[t_value] = probe
    feature_sets[t_value] = (train_features, train_labels, test_features, test_labels)
    probe_rows.append({
        "t": t_value,
        "effective noise std": BASE_STD * (1-t_value) / t_value,
        "training accuracy": train_accuracy,
        "held-out accuracy": test_accuracy,
        "95% interval low": interval_low,
        "95% interval high": interval_high,
        "shuffled mean": float(np.mean(shuffled_accuracies)),
        "shuffled std": float(np.std(shuffled_accuracies)),
        "shuffled 95th percentile": float(np.quantile(shuffled_accuracies, 0.95)),
        "shuffled max": float(np.max(shuffled_accuracies)),
    })

probe_table = pd.DataFrame(probe_rows)
display(probe_table.round(3))

fig, ax = plt.subplots(figsize=(7, 3.8))
lower_error = np.maximum(
    0.0, probe_table["held-out accuracy"] - probe_table["95% interval low"]
)
upper_error = np.maximum(
    0.0, probe_table["95% interval high"] - probe_table["held-out accuracy"]
)
ax.errorbar(
    probe_table["t"], probe_table["held-out accuracy"],
    yerr=np.vstack([lower_error, upper_error]), marker="o", capsize=3,
    color="#0072B2", label="held-out true-label probe",
)
ax.plot(probe_table["t"], probe_table["shuffled mean"], color="#D55E00", label="20 shuffled-label probes")
ax.fill_between(
    probe_table["t"],
    probe_table["shuffled mean"] - probe_table["shuffled std"],
    probe_table["shuffled mean"] + probe_table["shuffled std"],
    color="#D55E00", alpha=0.18,
)
ax.axhline(1 / len(CLASS_NAMES), color="#777777", ls="--", label="chance")
ax.set(xlabel="t (noise -> data)", ylabel="held-out accuracy", ylim=(0.25, 1.02), title="Class information across time")
ax.annotate("near chance", (0.10, probe_table.loc[0, "held-out accuracy"]), xytext=(8, 14), textcoords="offset points")
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.show()


In [ ]:
# Fit display PCA only on training activations, then transform held-out activations.
REFERENCE_T = 0.70
train_features, train_labels, test_features, test_labels = feature_sets[REFERENCE_T]
pca_display = fit_pca_projection(train_features)
projected_test = pca_display.transform(test_features)

fig, ax = plt.subplots(figsize=(6, 5))
plot_labeled_points(
    projected_test, test_labels, ax=ax,
    title=f"Held-out activations in train-fit PCA coordinates at t={REFERENCE_T}", alpha=0.35,
)
ax.legend(frameon=False)
plt.show()

reference_predictions = probes[REFERENCE_T].predict(test_features)
confusion = torch.bincount(
    test_labels * len(CLASS_NAMES) + reference_predictions,
    minlength=len(CLASS_NAMES) ** 2,
).reshape(len(CLASS_NAMES), len(CLASS_NAMES)).float()
confusion_fraction = confusion / confusion.sum(dim=1, keepdim=True).clamp_min(1)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
image = ax.imshow(confusion_fraction.cpu(), vmin=0, vmax=1, cmap="Blues")
for row in range(len(CLASS_NAMES)):
    for column in range(len(CLASS_NAMES)):
        value = float(confusion_fraction[row, column].cpu())
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", color="white" if value > 0.55 else "black")
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set(xlabel="predicted class", ylabel="true class", title=f"Held-out probe confusion at t={REFERENCE_T}")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

reference_direction = directions[REFERENCE_T]
cosine_table = pd.DataFrame([
    {"t": t_value, "cosine with t=0.70 direction": float(torch.dot(direction, reference_direction).cpu())}
    for t_value, direction in directions.items()
])
display(cosine_table.round(3))

def direction_statistics(direction):
    scores = test_features @ direction
    positive = scores[test_labels == TARGET_CLASS]
    negative = scores[test_labels != TARGET_CLASS]
    gap = positive.mean() - negative.mean()
    pooled_std = torch.sqrt(0.5 * (positive.var(unbiased=False) + negative.var(unbiased=False))).clamp_min(1e-8)
    order = torch.argsort(scores)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(1, scores.numel() + 1, device=device, dtype=torch.float32)
    n_positive = positive.numel()
    n_negative = negative.numel()
    auc = (
        ranks[test_labels == TARGET_CLASS].sum() - n_positive * (n_positive + 1) / 2
    ) / (n_positive * n_negative)
    return float(gap.cpu()), float((gap / pooled_std).cpu()), float(auc.cpu())

shuffled_directions = []
null_direction_rows = []
for null_index in range(50):
    set_seed(SEED + 4000 + null_index)
    shuffled_train_labels = train_labels[torch.randperm(train_labels.numel(), device=device)]
    direction = discriminant_direction(train_features, shuffled_train_labels, TARGET_CLASS)
    shuffled_directions.append(direction)
    raw_gap, normalized_gap, auc = direction_statistics(direction)
    null_direction_rows.append({
        "shuffle": null_index,
        "held-out raw gap": raw_gap,
        "held-out normalized gap": normalized_gap,
        "held-out AUC": auc,
    })

shuffled_direction = shuffled_directions[0]
true_raw_gap, true_normalized_gap, true_auc = direction_statistics(reference_direction)
null_direction_table = pd.DataFrame(null_direction_rows)
direction_summary = pd.DataFrame({
    "direction": ["true labels", "50 shuffled-label mean", "largest shuffled-label gap"],
    "normalized held-out gap": [
        true_normalized_gap,
        null_direction_table["held-out normalized gap"].mean(),
        null_direction_table["held-out normalized gap"].max(),
    ],
    "held-out AUC": [
        true_auc,
        null_direction_table["held-out AUC"].mean(),
        null_direction_table.loc[null_direction_table["held-out normalized gap"].idxmax(), "held-out AUC"],
    ],
})
display(direction_summary.round(3))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(null_direction_table["held-out normalized gap"], bins=12, color="#999999", alpha=0.75, label="50 shuffled-label directions")
ax.axvline(true_normalized_gap, color="#D55E00", lw=2.5, label="true-label direction")
ax.set(xlabel="normalized target-vs-rest gap on held-out activations", ylabel="count", title="Direction null distribution")
ax.legend(frameon=False)
plt.show()

null_99 = float(null_direction_table["held-out normalized gap"].quantile(0.99))
reference_row = probe_table.set_index("t").loc[REFERENCE_T]
probe_evidence = pd.DataFrame({
    "check": [
        "held-out probe accuracy is above 0.80",
        "held-out accuracy beats the shuffled-label 95th percentile by 0.20",
        "true direction exceeds 99th percentile of shuffled directions",
        "true-direction held-out AUC is above 0.80",
    ],
    "observed": [
        reference_row["held-out accuracy"],
        reference_row["held-out accuracy"] - reference_row["shuffled 95th percentile"],
        true_normalized_gap - null_99,
        true_auc,
    ],
    "supported": [
        reference_row["held-out accuracy"] >= 0.80,
        reference_row["held-out accuracy"] - reference_row["shuffled 95th percentile"] >= 0.20,
        true_normalized_gap > null_99,
        true_auc >= 0.80,
    ],
})
display(probe_evidence.round(3))
label_direction_separated = bool(probe_evidence["supported"].all())
print("held-out class-decodability supported:", label_direction_separated)
print("PCA is display-only; the held-out probe and null tests are the decoding evidence.")


## 2. Activation intervention

The target direction is covariance-aware: it separates target examples from
the remaining classes in hidden space. During an active time window, the model
adds a scaled unit direction to the selected hidden layer. The scale is relative
to the current feature RMS, so it remains comparable across time.

This is a genuine hidden-feature intervention, not sample-space attraction. It
is still simpler than NA-RFM: the paper uses RFM directions in image-model
activation tensors, while this notebook uses a linear discriminant direction in
one MLP layer.


In [ ]:
set_seed(SEED + 50)
x0 = sample_base(1400, device=device)
target_reference = sample_class(1400, TARGET_CLASS, device=device)

methods = {
    "baseline": base_velocity(model),
    "activation 0.0": make_activation_steered_velocity(
        model, reference_direction, strength=0.0, start=0.38, end=0.90
    ),
    "shuffled activation": make_activation_steered_velocity(
        model, shuffled_direction, strength=4.0, start=0.38, end=0.90
    ),
    "activation steering": make_activation_steered_velocity(
        model, reference_direction, strength=4.0, start=0.38, end=0.90
    ),
    "noise alignment": make_noise_aligned_velocity(
        model, stats, target_class=TARGET_CLASS, strength=1.6
    ),
    "gradient guidance": make_gradient_guided_velocity(
        model, stats, target_class=TARGET_CLASS, strength=3.0, start=0.15, end=0.92
    ),
    "noise + activation": make_combined_velocity(
        model, stats, reference_direction,
        target_class=TARGET_CLASS, noise_strength=1.6, activation_strength=4.0,
    ),
}

rows = []
paths = {}
for name, velocity in methods.items():
    path, seconds = timed_sample(velocity, x0, steps=56, method="heun")
    metrics = evaluate_samples(path[-1].to(device), target_reference, stats, target_class=TARGET_CLASS)
    rows.append({"method": name, **metrics, "seconds": seconds})
    paths[name] = path

comparison_table = pd.DataFrame(rows)
comparison_table["runtime / baseline"] = comparison_table["seconds"] / comparison_table.loc[0, "seconds"]
zero_activation_error = float(torch.max(torch.abs(paths["baseline"][-1] - paths["activation 0.0"][-1])))
assert zero_activation_error < 1e-6
assert model_state_digest(model) == checkpoint_digest_before

null_control_table = comparison_table[
    comparison_table["method"].isin(["baseline", "activation 0.0", "shuffled activation", "activation steering"])
]
method_table = comparison_table[
    comparison_table["method"].isin(["baseline", "noise alignment", "gradient guidance", "activation steering", "noise + activation"])
]
print(f"paired endpoint max difference, baseline vs activation 0.0: {zero_activation_error:.2e}")
print("checkpoint digest unchanged after every rollout: True")
display(null_control_table.round(4))
display(method_table.round(4))


In [ ]:
plot_trajectory_comparison(
    {
        "baseline": paths["baseline"],
        "shuffled direction": paths["shuffled activation"],
        "activation steering": paths["activation steering"],
        "noise + activation": paths["noise + activation"],
    },
    target_reference=target_reference[:700],
    n_lines=18,
)
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 5))
marker_by_method = {
    "baseline": "o", "shuffled activation": "s",
    "activation steering": "D", "noise alignment": "^", "gradient guidance": "v",
    "noise + activation": "P",
}
label_offsets = {
    "baseline": (-52, -12),
    "shuffled activation": (5, 8),
    "activation steering": (5, 4),
    "noise alignment": (5, 4),
    "gradient guidance": (5, 4),
    "noise + activation": (5, -2),
}
tradeoff_table = comparison_table[comparison_table["method"] != "activation 0.0"]
for _, row in tradeoff_table.iterrows():
    ax.scatter(row["target_swd"], row["target_rate"], s=80, marker=marker_by_method[row["method"]])
    ax.annotate(
        row["method"], (row["target_swd"], row["target_rate"]),
        xytext=label_offsets[row["method"]], textcoords="offset points", fontsize=9,
    )
ax.set(xlabel="SWD to target class (lower is better)", ylabel="target-class rate (higher is better)", title="Control-quality trade-off")
ax.grid(alpha=0.2)
plt.show()


In [ ]:
method_rows = comparison_table.set_index("method")
activation_gain = method_rows.loc["activation steering", "target_rate"] - method_rows.loc["baseline", "target_rate"]
shuffled_gain = method_rows.loc["shuffled activation", "target_rate"] - method_rows.loc["baseline", "target_rate"]
label_specific_margin = activation_gain - shuffled_gain
activation_swd_reduction = method_rows.loc["baseline", "target_swd"] - method_rows.loc["activation steering", "target_swd"]
best_control_method = comparison_table.loc[comparison_table["target_rate"].idxmax(), "method"]
best_target_match_method = comparison_table.loc[comparison_table["target_swd"].idxmin(), "method"]

intervention_evidence = pd.DataFrame({
    "check": [
        "activation-only target-rate gain at least 0.10",
        "activation-only SWD decreases",
        "true direction beats shuffled-direction gain by at least 0.05",
        "combined method has strongest target control",
    ],
    "observed": [activation_gain, activation_swd_reduction, label_specific_margin, best_control_method],
    "supported": [
        activation_gain >= 0.10,
        activation_swd_reduction > 0,
        label_specific_margin >= 0.05 and label_direction_separated,
        best_control_method == "noise + activation",
    ],
})
display(intervention_evidence.round(4))
print("strongest target control:", best_control_method)
print("lowest SWD to target examples:", best_target_match_method)
print("activation-only paired improvement supported:", bool(intervention_evidence.loc[:1, "supported"].all()))
label_specific_steering = bool(intervention_evidence.loc[2, "supported"])
print("label-specific activation steering supported:", label_specific_steering)
if not label_specific_steering:
    print("claim downgrade: the generator is sensitive to activation perturbations, but label specificity is not established.")


## 3. Further comparisons

To explore sensitivity, vary one factor at a time while keeping the paired
protocol fixed:

1. **Collection time:** learn directions at `t = 0.40, 0.55, 0.70, 0.85`.
2. **Intervention window:** early, middle, or late with fixed direction/strength.
3. **Activation strength:** at least three values with fixed direction/window.
4. **Method:** baseline, noise alignment, gradient, activation, and combined.
5. **Robustness:** repeat the decisive comparison over three initial-noise seeds.

The useful quantities to compare are target-class rate, target SWD, diversity
ratio, runtime, and paired trajectories. A method is not "best" without stating
the quality constraint used for comparison.

**Claim boundaries**

- Supported: class is linearly decodable from this hidden layer on held-out
  examples, especially at later `t`.
- Conditional: the intervention is label-specific only when the true direction
  beats the matched shuffled-label direction in the paired sampling test.
- Not supported: class lives in one unique direction, the feature is causally
  necessary, or this linear discriminant is the RFM component of full NA-RFM.

## Paper bridge

| NA-RFM component | Toy implementation | Important limitation |
|---|---|---|
| High-noise noise alignment | class Gaussian denoiser minus full Gaussian denoiser | 2D full covariance, not image PCA |
| Forward activation collection | labeled points corrupted to a reference `t` | one small MLP layer |
| Target direction | covariance-aware linear discriminant | not Recursive Feature Machine |
| Online activation edit | fixed feature direction over a time window | no U-Net feature map or amplification pass |
| Gradient-free inference | yes for noise/activation methods | gradient baseline still backpropagates |

**Questions for discussion**

1. Does probe accuracy predict steering success at the same collection time?
2. Over which times is the learned direction stable enough to reuse?
3. When does combining coarse high-noise and later activation control help?
4. Which conclusion from 2D would require a new experiment before claiming it for images?
